# LAK-6 — Schema evolution (Iceberg vs plain Parquet)

**Break -> Detect -> Fix -> Prove.** The source schema changes over time; a good table format
evolves **in metadata, without rewriting history**. We apply each change to an Iceberg table via
`ALTER TABLE` and read back to see the behavior, then contrast with a bare Parquet directory that
has no table metadata.

**Pre-requisite:** the unified Spark server is up (`make up`); this notebook connects via Spark
Connect. **Spark UI:** http://localhost:4040. **Laptop-safe:** a few hundred rows, all under
`.tmp/`; the Teardown cell drops the table and removes the Parquet dir (`make clean` clears the rest).

Iceberg tracks every column by a stable **field-id**, so rename/reorder are safe and old data reads
back correctly. See the companion [`README.md`](./README.md), the
[Spark-UI guide](../../docs/spark-ui-guide.md), and the [troubleshooting sheet](../../docs/troubleshooting.md).

In [ ]:
from common.spark_session import spark, display_df
from common.iceberg_meta import table_health, compare_health
from pyspark.sql import functions as F

T    = "iceberg_catalog.default.lak6_orders"   # Iceberg table (field-id tracked)
PARQ = ".tmp/lak6_parquet"                      # plain Parquet dir (positional, no metadata)
print("Spark UI:", "http://localhost:4040")
spark

## Step 0 - the v1 schema + a batch of "old" rows

`orders` v1 is `(order_id, customer_id INT, amount, status)`. We write 200 rows, then evolve the
schema *underneath* this already-written data so every read below crosses the change.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")

v1 = (spark.range(1, 201).withColumnRenamed("id", "order_id")
      .withColumn("customer_id", F.pmod(F.col("order_id"), F.lit(25)).cast("int"))
      .withColumn("amount", F.round(F.rand(seed=1) * 100, 2))
      .withColumn("status", F.lit("new")))
v1.writeTo(T).using("iceberg").create()

print("v1 schema:")
spark.table(T).printSchema()
print("rows:", spark.table(T).count())
display_df(spark.table(T), limit=5)

## 1. ADD COLUMN -> old rows read back NULL

A new field-id is assigned. Data written before the add doesn't carry that id, so those rows
project as `NULL` for the new column; no data is rewritten. A fresh row can populate it.

In [ ]:
spark.sql(f"ALTER TABLE {T} ADD COLUMN region STRING")
print("columns after ADD:", spark.table(T).columns)

null_region = spark.sql(f"SELECT COUNT(*) AS n FROM {T} WHERE region IS NULL").first()["n"]
print(f"old rows reading back NULL region: {null_region} (of {spark.table(T).count()})")

spark.sql(f"INSERT INTO {T} VALUES (201, 7, 42.0, 'new', 'EMEA')")
display_df(spark.sql(f"SELECT order_id, region FROM {T} WHERE order_id IN (1, 201)"))

## 2. RENAME COLUMN -> works by field-id, existing data intact

Rename changes only the *name* attached to an existing field-id; every value stays addressable.
This is the operation that "breaks" on positional Parquet (Section 7) but is trivial here.

In [ ]:
spark.sql(f"ALTER TABLE {T} RENAME COLUMN amount TO total_amount")
print("columns after RENAME:", spark.table(T).columns)

display_df(spark.sql(
    f"SELECT order_id, total_amount FROM {T} WHERE order_id IN (1, 2, 201) ORDER BY order_id"))
print("non-null total_amount values:",
      spark.sql(f"SELECT COUNT(*) AS n FROM {T} WHERE total_amount IS NOT NULL").first()["n"])

## 3. DROP COLUMN -> column hidden, data not rewritten

Drop removes the field-id from the *current* schema. The bytes remain in the old data files (no
rewrite), but the column is no longer projected.

In [ ]:
spark.sql(f"ALTER TABLE {T} DROP COLUMN status")
print("columns after DROP:", spark.table(T).columns)
print("'status' gone from schema:", "status" not in spark.table(T).columns)
display_df(spark.table(T), limit=5)

## 4. Widen type (INT -> BIGINT) allowed; narrowing rejected

`customer_id` was `INT`. Promoting to `BIGINT` is lossless, so Iceberg allows it and old INT values
read back as BIGINT. The reverse (`BIGINT -> INT`) could truncate data, so the spec forbids it -
we catch the rejection in a `try/except`.

In [ ]:
spark.sql(f"ALTER TABLE {T} ALTER COLUMN customer_id TYPE BIGINT")
ctype = [f.dataType.simpleString() for f in spark.table(T).schema if f.name == "customer_id"][0]
print("customer_id type after widen:", ctype)
print("old values read back at new type:",
      [r["customer_id"] for r in spark.sql(
          f"SELECT customer_id FROM {T} ORDER BY order_id LIMIT 3").collect()])

try:
    spark.sql(f"ALTER TABLE {T} ALTER COLUMN customer_id TYPE INT")
    print("UNEXPECTED: narrowing was allowed")
except Exception as e:
    print("\nNarrowing BIGINT -> INT rejected (as expected):")
    print("  ", str(e).splitlines()[0][:160])

## 5. Prove it - evolution was metadata-only (data files unchanged)

Every change above edited only metadata. `common.iceberg_meta.table_health` confirms the
**`data_files` count did not move** for a schema operation - Iceberg never rewrote the rows. We
also peek at an **older snapshot** (`VERSION AS OF`): schema is recorded per snapshot, so the first
snapshot still shows the v1 columns (pre-rename `amount`, since-dropped `status`).

In [ ]:
before = table_health(spark, T, "after evolution")
spark.sql(f"ALTER TABLE {T} ADD COLUMN note STRING")   # metadata-only
after  = table_health(spark, T, "after one more ALTER")
compare_health([before, after])
print("\ndata_files unchanged by the ALTER ->", before["data_files"] == after["data_files"])

first_snap = spark.sql(
    f"SELECT snapshot_id FROM {T}.snapshots ORDER BY committed_at LIMIT 1").first()["snapshot_id"]
v1_cols = spark.sql(f"SELECT * FROM {T} VERSION AS OF {first_snap}").columns
print(f"\nsnapshot {first_snap} columns:", v1_cols)
print("old snapshot still has 'amount' & 'status':",
      "amount" in v1_cols and "status" in v1_cols)

## 6. Contrast - plain Parquet is positional, with no table metadata

The same idea on a bare Parquet directory. We write **v1** files (`order_id, amount`), then **v2**
files with an added `region` to the **same path**. A naive `spark.read.parquet(path)` reconciles
**positionally** and mismatches; `.option("mergeSchema","true")` is required. Then a **rename**
(v3 files) still breaks it - Parquet matches by name/position, not field-id, so the rename becomes
two half-null columns.

In [ ]:
import shutil
shutil.rmtree(PARQ, ignore_errors=True)

# v1 files: (order_id, amount)
(spark.range(1, 101).withColumnRenamed("id", "order_id")
 .withColumn("amount", F.round(F.rand(seed=3) * 100, 2))
 .write.mode("overwrite").parquet(PARQ))
# v2 files (same path) add a 'region' column
(spark.range(101, 151).withColumnRenamed("id", "order_id")
 .withColumn("amount", F.round(F.rand(seed=4) * 100, 2))
 .withColumn("region", F.lit("EMEA"))
 .write.mode("append").parquet(PARQ))

naive = spark.read.parquet(PARQ)
print("naive read columns       :", naive.columns, " <- 'region' may be missing (positional)")
merged = spark.read.option("mergeSchema", "true").parquet(PARQ)
print("mergeSchema=true columns :", merged.columns)
print("v1 rows with NULL region :", merged.where("region IS NULL").count(),
      "(of", merged.count(), ")")

# v3 files 'rename' amount -> total_amount (same path)
(spark.range(151, 201).withColumnRenamed("id", "order_id")
 .withColumn("total_amount", F.round(F.rand(seed=5) * 100, 2))
 .withColumn("region", F.lit("APAC"))
 .write.mode("append").parquet(PARQ))
renamed = spark.read.option("mergeSchema", "true").parquet(PARQ)
print("\ncolumns after a 'rename' :", renamed.columns, " <- amount AND total_amount")
print("rows amount IS NULL       :", renamed.where("amount IS NULL").count())
print("rows total_amount IS NULL :", renamed.where("total_amount IS NULL").count())
print("Diagnosis: Parquet matched by NAME/position -> the rename split into two half-null columns;")
print("on Iceberg (Section 2) the same rename kept every value under one field-id.")

## Takeaways & "in real production..."

- **Schema changes are metadata-only on Iceberg/Delta** - ADD / RENAME / DROP / widen don't rewrite
  history (Section 5 proved `data_files` stayed flat), so they're cheap and safe even on huge tables.
- **Field-id tracking is why rename & reorder are safe** - the format tracks column *identity*, not
  name or position. Never depend on column ordinal.
- **Widening is allowed, narrowing is rejected** - promote types freely (`INT->BIGINT`); a narrowing
  needs an explicit, opt-in lossy rewrite.
- **Plain Parquet is positional with no table metadata** - evolving a bare directory needs
  `mergeSchema`, and even that can't survive a rename. This gap is what open table formats close.
- **Coordinate downstream:** in dbt this is `on_schema_change` (`fail`/`ignore`/`sync_all_columns`) -
  a non-nullable add or a rename can still break an incremental model. That's **DBT-5**.

## Teardown

Drop the Iceberg table and remove the Parquet dir. `make clean` clears everything under `.tmp/`.

In [ ]:
import shutil
spark.sql(f"DROP TABLE IF EXISTS {T}")
shutil.rmtree(PARQ, ignore_errors=True)
print("Dropped", T, "and removed", PARQ, "(`make clean` clears all of .tmp).")